# SIEWS+ 5.0 — Stage 3: Fire & Smoke Detection
## Transfer Learning with YOLOv8n

**Target classes:** `fire`, `smoke`

---

## Cara Pakai

| Environment | Langkah |
|---|---|
| **Kaggle** | 1. Upload notebook ini<br>2. Attach dataset dari /kaggle/input<br>3. Enable GPU P100/T4<br>4. Run All |
| **Colab** | 1. Upload notebook<br>2. Mount Google Drive<br>3. Attach dataset<br>4. Run All |
| **Local** | `python train_stage3_fire_smoke.py` (gunakan file .py) |

## 1. Setup Environment

In [ ]:
import os
import subprocess

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    pass

print(f'Kaggle: {IS_KAGGLE} | Colab: {IS_COLAB}')

# Install ultralytics
subprocess.run(['pip', 'install', '-U', 'ultralytics', '-q'], check=True)

import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = '0'
else:
    print('No GPU — using CPU (akan sangat lambat!)')
    DEVICE = 'cpu'

## 2. Konfigurasi Path

In [ ]:
from pathlib import Path

# =====================================================
# SESUAIKAN path di sini jika perlu
# =====================================================
if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_DIR    = Path('/kaggle/input')
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/migas-siews')
    INPUT_DIR    = PROJECT_ROOT / 'dataset'
else:
    # Local Windows: auto-detect dari lokasi file notebook
    _nb_path = Path('/home/atokdins/Atok/migas-siews/training/train_stage3_fire_smoke_v2.ipynb')
    PROJECT_ROOT = _nb_path.parent.parent  # naik 2 level dari training/
    INPUT_DIR    = PROJECT_ROOT / 'dataset'

OUTPUT_DIR = PROJECT_ROOT / 'training' / 'runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'INPUT_DIR    : {INPUT_DIR}')
print(f'OUTPUT_DIR   : {OUTPUT_DIR}')

## 3. Setup Dataset

In [ ]:
import zipfile
import shutil

# Fungsi untuk cari dataset
def find_dataset_root():
    """Cari folder dataset yang punya struktur train/images."""
    for search_root in [INPUT_DIR, PROJECT_ROOT]:
        if not search_root.exists():
            continue
        for pattern in ['stage3', 'Fire and Smoke', 'fire-smoke', 'Fire_and_Smoke']:
            candidate = search_root / pattern
            if (candidate / 'train' / 'images').exists():
                return candidate
        # Fallback: cari recursive
        train_dirs = list(search_root.rglob('train/images'))
        if train_dirs:
            train_dirs.sort(key=lambda p: len(p.parts))
            return train_dirs[0].parent.parent
    return None

def find_zip():
    """Cari fire/smoke dataset ZIP."""
    zips = list(INPUT_DIR.rglob('*.zip'))
    fire_zips = [z for z in zips if 'fire' in z.name.lower() or 'smoke' in z.name.lower()]
    return fire_zips[0] if fire_zips else None

# Resolve dataset
dataset_root = find_dataset_root()
stage3_zip   = find_zip()
EXTRACT_DIR  = OUTPUT_DIR / 'stage3_dataset'

print(f'Dataset ZIP  : {stage3_zip}')
print(f'Dataset folder: {dataset_root}')

# Extract jika perlu
if dataset_root is None and stage3_zip and stage3_zip.exists():
    print(f'Extracting {stage3_zip.name} ...')
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    with zipfile.ZipFile(stage3_zip, 'r') as zf:
        zf.extractall(OUTPUT_DIR)
    extracted_roots = list(OUTPUT_DIR.rglob('train/images'))
    if extracted_roots:
        extracted_roots.sort(key=lambda p: len(p.parts))
        dataset_root = extracted_roots[0].parent.parent

# Symlink ke folder dataset
if dataset_root and not EXTRACT_DIR.exists():
    try:
        EXTRACT_DIR.symlink_to(dataset_root.resolve())
        print(f'Symlinked to: {dataset_root}')
    except:
        shutil.copytree(dataset_root, EXTRACT_DIR)

# Resolve symlink
if EXTRACT_DIR.is_symlink():
    EXTRACT_DIR = EXTRACT_DIR.resolve()

print(f'EXTRACT_DIR : {EXTRACT_DIR}')

## 4. Validasi Struktur Dataset

In [ ]:
# Cek struktur folder
train_img = EXTRACT_DIR / 'train' / 'images'
val_img   = EXTRACT_DIR / 'val' / 'images'
if not val_img.exists():
    val_img = EXTRACT_DIR / 'valid' / 'images'

val_split = 'valid' if (EXTRACT_DIR / 'valid' / 'images').exists() else 'val'

print('--- Dataset Structure ---')
for split, img_dir in [('train', train_img), ('val', val_img)]:
    if img_dir.exists():
        n_img = len(list(img_dir.glob('*.*')))
        lbl_dir = img_dir.parent.parent / 'labels'
        n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f'[{split:6s}] images={n_img:5d}  labels={n_lbl:5d}')
    else:
        print(f'[{split:6s}] NOT FOUND: {img_dir}')

## 5. Label Audit

> **Kritis:** Verifikasi bahwa `0=fire` dan `1=smoke` di dataset label. Jika terbalik, model akan salah klasifikasi.

In [ ]:
from collections import Counter

# SATU SUMBER KEBENARAN — jangan ubah kecuali audit membuktikan lain
CANONICAL_CLASS_NAMES  = {0: 'fire', 1: 'smoke'}
CANONICAL_CLASS_LIST   = ['fire', 'smoke']  # YOLO butuh LIST, bukan dict!
CANONICAL_CLASS_COLORS = {0: (255, 80, 0), 1: (180, 180, 180)}  # RGB

print('--- Label Audit ---')
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
cls_counter = Counter()
n_audited = 0

if train_lbl_dir.exists():
    for lbl_file in train_lbl_dir.glob('*.txt'):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if parts:
                cls_counter[int(parts[0])] += 1
        n_audited += 1

print(f'Files audited: {n_audited}')
for cls_id, count in sorted(cls_counter.items()):
    name = CANONICAL_CLASS_NAMES.get(cls_id, f'UNKNOWN-{cls_id}')
    print(f'  cls {cls_id} ({name:6s}): {count:6d} boxes')

unexpected = [k for k in cls_counter if k not in CANONICAL_CLASS_NAMES]
if unexpected:
    print(f'\nWARNING: Unexpected class IDs: {unexpected}')
    print('Please verify your dataset labels!')
else:
    print('\n✅ Class IDs valid (0=fire, 1=smoke)')

## 6. Buat Custom YAML

In [ ]:
import yaml

custom_yaml = OUTPUT_DIR / 'stage3_fire_smoke.yaml'
yaml_content = {
    'path' : str(EXTRACT_DIR),
    'train': 'train/images',
    'val'  : f'{val_split}/images',
    'test' : 'test/images' if (EXTRACT_DIR / 'test' / 'images').exists() else f'{val_split}/images',
    'nc'   : 2,
    'names': CANONICAL_CLASS_LIST,  # LIST, bukan dict!
}

with open(custom_yaml, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f'Custom YAML: {custom_yaml}')
print('\n--- YAML Content ---')
print(custom_yaml.read_text())

# Verify
_verify = yaml.safe_load(custom_yaml.read_text())
assert _verify['names'] == CANONICAL_CLASS_LIST, f'YAML names mismatch: {_verify["names"]}'
assert _verify['nc'] == 2
print('\n✅ YAML verified: names =', _verify['names'])

## 7. Visualisasi Sample Dataset

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

train_img_dir = EXTRACT_DIR / 'train' / 'images'
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'

all_imgs = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))
samples = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            cls = int(cls)
            x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
            color = CANONICAL_CLASS_COLORS.get(cls, (0, 255, 0))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, f'ID:{cls} {CANONICAL_CLASS_NAMES.get(cls, "?")}',
                       (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    ax.imshow(img); ax.axis('off'); ax.set_title(img_path.name[:20])

plt.suptitle('Sample Dataset — Fire & Smoke', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'dataset_samples.png'), dpi=100)
plt.show()
print('Samples visualized!')

## 8. Load Base Model YOLOv8n

In [ ]:
from ultralytics import YOLO

# Download pretrained YOLOv8n (COCO, 80 classes)
base_model = YOLO('yolov8n.pt')

print(f'Model loaded: {base_model.model_name}')
print(f'Base classes: {len(base_model.names)} (COCO)')
print('\nSiap untuk fine-tuning ke 2 kelas: fire, smoke')

## 9. Konfigurasi Training

In [ ]:
# Auto-adjust batch size berdasarkan VRAM GPU
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_mem_gb > 14:
        BATCH = 16
    elif gpu_mem_gb > 7:
        BATCH = 8
    else:
        BATCH = 4
else:
    BATCH = 4

EPOCHS   = 80
IMGSZ    = 640
PATIENCE = 15
FREEZE   = 10   # freeze backbone — transfer learning

print(f'Config: epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'        freeze={FREEZE} layers (transfer learning mode)')

## 10. Training (Transfer Learning)

### Penjelasan Hyperparameter:

| Parameter | Nilai | Alasan |
|---|---|---|
| `epochs` | 80 | Cukup untuk konvergensi fire/smoke |
| `batch` | auto (8-16) | Sesuai VRAM GPU Kaggle |
| `imgsz` | 640 | Resolusi standar YOLO |
| `patience` | 15 | Early stopping jika tidak improve |
| `freeze` | 10 | Freeze 10 layer pertama — transfer learning |
| `hsv_h/s/v` | tinggi | Fire/smoke sangat bergantung warna |
| `mosaic` | 1.0 | Gabungkan 4 gambar — perbanyak variasi |
| `flipud` | 0.1 | Api bisa dari segala arah |
| `optimizer` | `AdamW` | Lebih stabil untuk fine-tuning |

In [ ]:
results = base_model.train(
    data    = str(custom_yaml),
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    patience= PATIENCE,
    freeze  = FREEZE,
    pretrained = True,

    # Project output
    project  = str(OUTPUT_DIR / 'runs'),
    name     = 'stage3_fire_smoke',
    exist_ok = True,

    # Augmentasi — penting untuk fire/smoke
    augment  = True,
    mosaic   = 1.0,
    mixup    = 0.1,
    copy_paste = 0.1,
    hsv_h    = 0.03,
    hsv_s    = 0.9,
    hsv_v    = 0.5,
    fliplr   = 0.5,
    flipud   = 0.1,
    degrees  = 5.0,
    scale    = 0.6,
    translate= 0.1,
    erasing  = 0.3,

    # Optimizer
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    weight_decay  = 0.0005,
    warmup_epochs = 3,

    # Save & logging
    save_period = 10,
    verbose     = True,
    plots       = True,
)

## 11. Evaluasi Hasil Training

In [ ]:
print('=' * 50)
print('HASIL TRAINING — Stage 3 Fire/Smoke')
print('=' * 50)

metrics = results.results_dict
map50    = metrics.get('metrics/mAP50(B)', 0)
map5095  = metrics.get('metrics/mAP50-95(B)', 0)
prec     = metrics.get('metrics/precision(B)', 0)
rec      = metrics.get('metrics/recall(B)', 0)

print(f'mAP50       : {map50:.4f}  (target: > 0.85)')
print(f'mAP50-95    : {map5095:.4f}')
print(f'Precision   : {prec:.4f}  (target: > 0.80)')
print(f'Recall      : {rec:.4f}  (target: > 0.80)')
print()

if map50 >= 0.85:
    print('✅ Model sudah baik! mAP50 >= 0.85')
elif map50 >= 0.70:
    print('⚠️  Model cukup baik. Coba tambah epochs atau lebih banyak data.')
else:
    print('❌ Model kurang baik. Lihat saran di bawah.')

In [ ]:
# Tampilkan grafik training
run_dir = OUTPUT_DIR / 'runs' / 'stage3_fire_smoke'

from IPython.display import Image, display

plot_files = [
    run_dir / 'results.png',
    run_dir / 'confusion_matrix.png',
    run_dir / 'PR_curve.png',
]

for pf in plot_files:
    if pf.exists():
        print(f'--- {pf.name} ---')
        display(Image(str(pf), width=800))

## 12. Validasi pada Val Set

In [ ]:
# Load best model
best_pt = run_dir / 'weights' / 'best.pt'
print(f'Loading best model: {best_pt}')

best_model = YOLO(str(best_pt))

# Patch & Re-save: satu-satunya cara aman paksa metadata class
loaded_names = {int(k): v for k, v in dict(best_model.names).items()}
print(f'Model names (dari weight): {loaded_names}')

if loaded_names != CANONICAL_CLASS_NAMES:
    print(f'WARNING: Class order berbeda dari canonical! {loaded_names}')
    print(f'         → Melakukan patch dan re-save...')
    best_model.model.names = CANONICAL_CLASS_NAMES
    patched_pt = run_dir / 'weights' / 'best_patched.pt'
    best_model.save(str(patched_pt))
    best_model = YOLO(str(patched_pt))
    print(f'Patched model saved & reloaded: {patched_pt}')
else:
    print(f'✅ Class order OK: {loaded_names}')

# Konfirmasi setelah patch
final_names = {int(k): v for k, v in dict(best_model.names).items()}
print(f'Final model.names: {final_names}')
assert final_names == CANONICAL_CLASS_NAMES, f'Patch gagal: {final_names}'

# Validasi
val_results = best_model.val(data=str(custom_yaml), device=DEVICE, verbose=True)
print(f'\nVal mAP50   : {val_results.box.map50:.4f}')
print(f'Val mAP50-95: {val_results.box.map:.4f}')

## 13. Inference Test

In [ ]:
# Pastikan model sudah di-patch
assert {int(k): v for k, v in dict(best_model.names).items()} == CANONICAL_CLASS_NAMES, \
    "best_model.names tidak sesuai CANONICAL! Jalankan ulang Cell 12."

# Ambil test images
test_img_dir = EXTRACT_DIR / 'test' / 'images'
if not test_img_dir.exists():
    test_img_dir = EXTRACT_DIR / val_split / 'images'

all_test = list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png'))
test_imgs = random.sample(all_test, min(6, len(all_test)))

TEST_CONF = 0.15  # smoke diffuse, conf rendah untuk audit visual

# Warna BGR (cv2)
PRED_COLORS_BGR = {
    0: (0, 140, 255),    # fire → orange
    1: (180, 180, 180),  # smoke → gray
}
GT_COLOR_BGR    = (0, 255, 80)  # ground truth → green

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, test_imgs):
    pred    = best_model.predict(str(img_path), conf=TEST_CONF, verbose=False)[0]
    img_bgr = cv2.imread(str(img_path))
    h, w    = img_bgr.shape[:2]
    boxes   = pred.boxes
    label_parts = []

    # Ground truth (kotak hijau tipis)
    lbl_path = test_img_dir.parent.parent / 'labels' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if not parts: continue
            cls_gt = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            gx1 = int((xc - bw/2) * w); gy1 = int((yc - bh/2) * h)
            gx2 = int((xc + bw/2) * w); gy2 = int((yc + bh/2) * h)
            gt_name = CANONICAL_CLASS_NAMES.get(cls_gt, f'cls_{cls_gt}')
            cv2.rectangle(img_bgr, (gx1, gy1), (gx2, gy2), GT_COLOR_BGR, 1)
            cv2.putText(img_bgr, f'GT:{gt_name}', (gx1, max(gy1 - 18, 14)),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, GT_COLOR_BGR, 1)

    # Predictions
    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            name  = CANONICAL_CLASS_NAMES.get(cls_id, f'UNKNOWN-{cls_id}')
            color = PRED_COLORS_BGR.get(cls_id, (0, 255, 0))
            cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img_bgr, f'{name} {conf:.0%}', (x1, max(y1 - 6, 18)),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
            label_parts.append(f'ID:{cls_id}={name}({conf:.0%})')

    annotated = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.axis('off')
    preds_str = ', '.join(label_parts) if label_parts else '— no detection —'
    ax.set_title(preds_str[:40], fontsize=8)

plt.suptitle('Inference Test — Best Model\n(oranye=fire, abu=smoke, hijau=GT)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'inference_test.png'), dpi=100)
plt.show()
print('\nInference test selesai!')

## 14. Simpan Model ke Backend

In [ ]:
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

# Simpan ulang best model dengan metadata class yang sudah dipaksa benar
export_model = best_model
export_model.names = CANONICAL_CLASS_NAMES

if IS_KAGGLE:
    # Di Kaggle: salin ke /kaggle/working agar bisa di-download
    out_best = Path('/kaggle/working/best_stage3_fire_smoke.pt')
    out_last = Path('/kaggle/working/last_stage3_fire_smoke.pt')
    export_model.save(str(out_best))
    shutil.copy2(last_pt, out_last)
    print(f'✅ Saved (Kaggle output):')
    print(f'   {out_best}  ({out_best.stat().st_size/1e6:.1f} MB)')
    print(f'   {out_last}  ({out_last.stat().st_size/1e6:.1f} MB)')
    print()
    print('Langkah selanjutnya:')
    print('  1. Download best_stage3_fire_smoke.pt dari Output tab Kaggle')
    print('  2. Salin ke: backend/models/best_stage3_fire_smoke.pt')
    print('  3. Restart backend')
else:
    # Lokal: salin langsung ke backend/models
    models_dir = PROJECT_ROOT / 'backend' / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    out_best = models_dir / 'best_stage3_fire_smoke.pt'
    out_last = models_dir / 'last_stage3_fire_smoke.pt'
    export_model.save(str(out_best))
    shutil.copy2(last_pt, out_last)
    print(f'✅ Saved to backend: {out_best}  ({out_best.stat().st_size/1e6:.1f} MB)')
    print(f'   Also saved: {out_last}  ({out_last.stat().st_size/1e6:.1f} MB)')

## 15. Saran jika mAP Rendah

| Masalah | Solusi |
|---|---|
| mAP50 < 0.70 | Tambah data dari Roboflow: cari `fire detection` / `smoke detection` |
| Banyak false positive | Tambah gambar negatif (tanpa api/asap) ke dataset |
| Detection NONE saat test | Turunkan `conf` threshold ke 0.15 |
| Training lambat | Kurangi `batch=8`, atau upgrade ke GPU T4 x2 di Kaggle |
| Loss tidak turun | Kurangi `lr0=0.0005`, naikkan `warmup_epochs=5` |
| Overfitting | Naikkan `weight_decay=0.001`, tambah `dropout=0.1` |

### Dataset tambahan yang recommended (gratis di Roboflow):
- `D-Fire Dataset` — 21k gambar api & asap outdoor
- `Fire Detection v2` — 5k gambar indoor & outdoor
- `Wildfire Smoke` — khusus asap hutan